In [ ]:
##alles was hier kommt sind nur rundungsfunktionen und eine voreinstellung für die Achsenbeschriftung




using DataFrames
using Measurements
import Measurements: value, uncertainty
using Statistics



using Random, Statistics

function first_significant_digit(x::Real)
    x == 0 && return 0  # Sonderfall: Wenn x genau 0 ist, gibt es keine signifikante Stelle, also gib 0 zurück.

    absx = abs(x)  # Der Betrag der Zahl wird genommen, damit negative Vorzeichen ignoriert werden.

    # Berechne die Zehnerpotenz, in der sich die erste signifikante Stelle befindet.
    # log10(absx) gibt den dekadischen Logarithmus. floor(Int, ...) rundet nach unten auf ganze Zahl.
    exponent = floor(Int, log10(absx))

    # Teile absx durch 10^exponent, um die Zahl in den Bereich [1,10) zu bringen
    # Beispiel: aus 456 → 4.56, aus 0.00456 → 4.56
    significand = absx / 10.0^exponent

    # Jetzt holen wir die ganzzahlige Ziffer vor dem Komma, das ist die erste signifikante Ziffer
    return floor(Int, significand)
end


"""
    round_measurement(m::Measurement; sigdigits_error::Int=1)

Rundet erst die Unsicherheit von `m` auf `sigdigits_error` signifikante Stellen
und rundet dann den Messwert auf dieselbe Genauigkeit (Dezimalstellen),
liefert einen neuen `Measurement`. Sigdigits_error wird automatisch auf den richtigen wert gesetzt durch if bedingung 
wenn first sig digit =1 oder 2, wird eine weitere stelle hinzugenommen.

input: ein measurement, output: ein measurement. Kann punktweise angewendet werden
"""
function round_measurement(m::Measurement)
    # 1) Unsicherheit extrahieren und auf sigdigits_error sig. Stellen runden
    u = uncertainty(m)
    if first_significant_digit(u)<3
        sigdigits_error=2
    else
        sigdigits_error=1
    end
    #println(sigdigits_error)

    u_r = round(u; sigdigits=sigdigits_error)

    
    if u_r==0 #wenn gauß versagt, normalverteilung und stw
        

    # Ziehe N Stichproben aus einer Normalverteilung und berechne KINETISCHE ENERGIE!!!
    p_samples = randn(100_000) .* 0.05  # p ~ N(0, σp=0.05) kein plan warum, aber die 0.05 sind auf jkeden fall der fehler vorher

    # Berechne kinetische Energie für jede Probe
    E_samples = p_samples.^2 ./ (2)

    # Mittelwert und Standardabweichung der Energie
    E_mean = mean(E_samples)
    E_std = std(E_samples)
    
    u_r = round(E_std; sigdigits=sigdigits_error)
        #println(u_r)
    end
    
    
    
    
    
    # 2) Anzahl Dezimalstellen bestimmen:
    #    Wenn u_r = x * 10^e  (mit 1 ≤ x < 10), dann ist e = floor(log10(u_r))
    #    und wir benötigen -e  Dezimalstellen (für e ≤ 0)
    e = floor(Int, log10(u_r))
       # println(e)
    if first_significant_digit(u)<3
        dec = max(0, -e) +1
    else
        dec = max(0, -e)
    end
    #dec = max(0, -e)
    #println(dec, log(12,u_r))

    # 3) Wert runden und neuen Measurement erstellen
    v_r = round(value(m); digits=dec)
    return measurement(v_r, u_r)
end

# Beispiel
m = measurement(0.023456, 0.0120236789)
println("Original: ", value(m))                  # 1.23456 ± 0.06789
m2 = round_measurement(m)
println("Gerundet: ", value(m2))  # z.B. 1.23(7) → ±0.07, Wert 1.23




[ Info: Precompiling DataFrames [a93c6f00-e57d-5684-b7b6-d8193f3e46c0] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [10]:

function generate_reflections(hmax,nth)
    sc  = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
    bcc = Dict{Int, Vector{Tuple{Int,Int,Int}}}()
    fcc = Dict{Int, Vector{Tuple{Int,Int,Int}}}()

    for h in 0:hmax, k in 0:hmax, l in 0:hmax

        N = h^2 + k^2 + l^2
        hkl = (h,k,l)

        # --- SC: alle erlaubten Reflexe ---
        if length(sc) == nth
            if N < maximum(keys(sc))
                delete!(sc, maximum(keys(sc)))
            end
        end 

        if length(fcc) == nth
            if N < maximum(keys(fcc))
                delete!(fcc, maximum(keys(fcc)))
            end
        end 

        if length(bcc) == nth
            if N < maximum(keys(bcc))
                delete!(bcc, maximum(keys(bcc)))
            end
        end 


        if length(sc) < nth
            push!(get!(sc, N, Vector{Tuple{Int,Int,Int}}()), hkl)
        end

        # --- BCC: h+k+l gerade ---
        if length(bcc) < nth
            if (h + k + l) % 2 == 0
                push!(get!(bcc, N, Vector{Tuple{Int,Int,Int}}()), hkl) 
            end
        end

        # --- FCC: alle h,k,l entweder alle gerade oder alle ungerade ---
        if length(fcc) < nth
            if (h % 2 == k % 2 == l % 2)
                push!(get!(fcc, N, Vector{Tuple{Int,Int,Int}}()), hkl)
            end
        end
    end
    sc_sorted = [(N, sc[N]) for N in sort(collect(keys(sc)))]
    bcc_sorted = [(N, bcc[N]) for N in sort(collect(keys(bcc)))]
    fcc_sorted = [(N, fcc[N]) for N in sort(collect(keys(fcc)))]
    return sc_sorted, bcc_sorted, fcc_sorted
end


generate_reflections (generic function with 1 method)

In [58]:
function anglestheo(n)
    rfl=generate_reflections(n,n)[3]
    println(rfl)
    α = zeros(n)
    for i in 1:n
        if rfl[i][2][1] == (0,0,0) 
        α[i] = 0
        print("hi")
        else
        α[i] = acos((rfl[i][2][1][1]+rfl[i][2][1][2]+rfl[i][2][1][3])/(sqrt(3)* sqrt((rfl[i][2][1][1]^2+rfl[i][2][1][2]^2+rfl[i][2][1][3]^2)))-0.00000000002)/pi*180
        end
    end
return round_measurement.(measurement.(α,0.001))
end
#also ich berechne hier einfach nur die winkel zwischen ebenen mit millerindizes. das entspricht den winkeln ihrer normalen also der rez. gittervektoren
# wegen kubischen gitter kann ich einfaches SP benutzen
#
#dannach schaue ich mir die gemessenen reflexe an und kriege damit die richtung von k´(ich sage k, k´haben selben betrag). aus k-k´kann ich die richtung von g berechnen.
#mit diesem g kann ich dann den winkel zu k berechnen, wobei k parallel zu 1 1 1 ist und die reflexe zuordnen. ich kann nur die wiunkel zwischen k und deltak berechnen
#aber ich kann nicht die richtung von deltak im gitterkoordinatensystem bestimmen, darum das winkelgerechne

anglestheo (generic function with 1 method)

In [12]:
using LinearAlgebra

In [41]:
function anglesprax(th, ph,th111,ph111)
    th = th .* pi/180
    ph = ph .* pi/180
    kk(x,y)=[(sin(x)*cos(y))';
     (sin(x).*sin(y))';
      (cos(x)')]
    #println(kk.(th,ph))
    kkvec=kk.(th,ph)
    kvec = kk(th111*pi/180, ph111*pi/180)
    g = zeros(3,length(th))
    #println(g)
    h= zeros(length(th))
    kg=(kvec-[0,0,-1])/norm(kvec-[0,0,-1])
    for i in 1:length(th)
        #println(kvec)
        g[:, i] =  kkvec[i]-[0,0,-1]
        #println(g[:, i])
        g[:, i]=g[:, i]/norm(g[:, i])
        h[i] = acos(kg' *g[:, i])/pi*180
    end
    return h
end

anglesprax (generic function with 2 methods)

In [ ]:
function anglespraxy(th, ph,th111,ph111)
    th = th .* pi/180
    ph = ph .* pi/180
    kk(x,y)=[(cos(x)*sin(y))';
     (cos(x).*cos(y))';
      (sin(x)')]
    #println(kk.(th,ph))
    kkvec=kk.(th,ph)
    kvec = kk(th111*pi/180, ph111*pi/180)
    g = zeros(3,length(th))
    #println(g)
    h= zeros(length(th))
    kg=(kvec-[0,-1,0])/norm(kvec-[0,-1,0])
    for i in 1:length(th)
        #println(kvec)
        g[:, i] =  kkvec[i]-[0,-1,0]
        #println(g[:, i])
        g[:, i]=g[:, i]/norm(g[:, i])
        h[i] = acos(kg' *g[:, i])/pi*180
    end
    return h
end

anglespraxy (generic function with 1 method)

In [25]:
th, ph = [0.0, 90, 60], [0.0, 0, 60]
anglesprax(th, ph, -13, 15)


3-element Vector{Float64}:
  6.50000000000002
 51.30003615638831
 34.87162858219098

In [26]:
[1 0 0] - [2 3 4]

1×3 Matrix{Int64}:
 -1  -3  -4

In [42]:
ph = -1 .*[5.5, 2, -8, 10, 6.5, 2, -4, -13, -25, 20.5, 8, -6.5, -5.5, -16, -18]
th = [28, 26, 26, 18, 17.5, 16.5, 16, 15, 13.5, 8, 6.5, 7, 6, 6, 1.5]


15-element Vector{Float64}:
 28.0
 26.0
 26.0
 18.0
 17.5
 16.5
 16.0
 15.0
 13.5
  8.0
  6.5
  7.0
  6.0
  6.0
  1.5

In [47]:
anglesprax([13],[15], 13, 15)

1-element Vector{Float64}:
 0.0

In [53]:
anglespraxy([0],[0], 0, 0)

1-element Vector{Float64}:
 0.0

In [45]:
generate_reflections(14,14) 

([(0, [(0, 0, 0)]), (1, [(0, 0, 1), (0, 1, 0), (1, 0, 0)]), (2, [(0, 1, 1), (1, 0, 1), (1, 1, 0)]), (3, [(1, 1, 1)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (5, [(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]), (6, [(1, 1, 2), (1, 2, 1), (2, 1, 1)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (9, [(0, 0, 3), (0, 3, 0), (1, 2, 2), (2, 1, 2), (2, 2, 1), (3, 0, 0)]), (10, [(0, 1, 3), (0, 3, 1), (1, 0, 3), (1, 3, 0), (3, 0, 1), (3, 1, 0)]), (11, [(1, 1, 3), (1, 3, 1), (3, 1, 1)]), (12, [(2, 2, 2)]), (13, [(0, 2, 3), (0, 3, 2), (2, 0, 3), (2, 3, 0), (3, 0, 2), (3, 2, 0)]), (14, [(3, 2, 1)])], [(0, [(0, 0, 0)]), (2, [(0, 1, 1), (1, 0, 1), (1, 1, 0)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (6, [(1, 1, 2), (1, 2, 1), (2, 1, 1)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (10, [(0, 1, 3), (0, 3, 1), (1, 0, 3), (1, 3, 0), (3, 0, 1), (3, 1, 0)]), (12, [(2, 2, 2)]), (14, [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]), (16, [(0, 0, 4), (0, 4, 0), (4, 0, 0)]), (

In [46]:
anglespraxy(ph, th, 13, 15)

15-element Vector{Float64}:
 11.450847433002332
  9.415833509576897
  6.029400236999323
 11.717347720331357
  9.931531671216721
  7.612951923796617
  4.57368796178124
  1.2074182697257333e-6
  6.1064034452909155
 17.193909609853723
 11.352110869115805
  5.10735913039309
  5.810156392050283
  4.65838661917522
  7.0467493748274705

In [56]:
anglesprax(ph, th, 13, 15)

15-element Vector{Float64}:
  9.200249132102705
  7.484049503754279
  2.6838199675003542
 11.496113465569444
  9.747933160687206
  7.499702812836956
  4.500439389637866
  0.0
  6.00460044848175
 16.72012332687271
 10.472694034541584
  3.312521239627887
  3.8081292275942142
  1.8771539874916012
  3.0763325869867666

In [59]:
anglestheo(20)

[(0, [(0, 0, 0)]), (3, [(1, 1, 1)]), (4, [(0, 0, 2), (0, 2, 0), (2, 0, 0)]), (8, [(0, 2, 2), (2, 0, 2), (2, 2, 0)]), (11, [(1, 1, 3), (1, 3, 1), (3, 1, 1)]), (12, [(2, 2, 2)]), (16, [(0, 0, 4), (0, 4, 0), (4, 0, 0)]), (19, [(1, 3, 3), (3, 1, 3), (3, 3, 1)]), (20, [(0, 2, 4), (0, 4, 2), (2, 0, 4), (2, 4, 0), (4, 0, 2), (4, 2, 0)]), (24, [(2, 2, 4), (2, 4, 2), (4, 2, 2)]), (27, [(1, 1, 5), (1, 5, 1), (3, 3, 3), (5, 1, 1)]), (32, [(0, 4, 4), (4, 0, 4), (4, 4, 0)]), (35, [(1, 3, 5), (1, 5, 3), (3, 1, 5), (3, 5, 1), (5, 1, 3), (5, 3, 1)]), (36, [(0, 0, 6), (0, 6, 0), (2, 4, 4), (4, 2, 4), (4, 4, 2), (6, 0, 0)]), (40, [(0, 2, 6), (0, 6, 2), (2, 0, 6), (2, 6, 0), (6, 0, 2), (6, 2, 0)]), (43, [(3, 3, 5), (3, 5, 3), (5, 3, 3)]), (44, [(2, 2, 6), (2, 6, 2), (6, 2, 2)]), (48, [(4, 4, 4)]), (51, [(1, 1, 7), (1, 5, 5), (1, 7, 1), (5, 1, 5), (5, 5, 1), (7, 1, 1)]), (59, [(7, 3, 1)])]
hi

LoadError: UndefVarError: `round_measurement` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
#unorientierte phi
ph = -1 .*[5.5, 2, -8, 10, 6.5, 2, -4, -13, -25, 20.5, 8, -6.5, -5.5, -16, -18]
th = [28, 26, 26, 18, 17.5, 16.5, 16, 15, 13.5, 8, 6.5, 7, 6, 6, 1.5]


15-element Vector{Float64}:
 28.0
 26.0
 26.0
 18.0
 17.5
 16.5
 16.0
 15.0
 13.5
  8.0
  6.5
  7.0
  6.0
  6.0
  1.5